In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from functools import lru_cache

# Optional parallelization (used only if needed)
from joblib import Parallel, delayed

# ============================
# IMPORT YOUR PHYSICS MODEL
# ============================
import photop as model


# ============================
# LOAD DATA
# ============================
data = pd.read_csv("selected_omega_dataset.csv")

W_data = data["W"].values
sigma_data = data["dsigdt (mu barn/GeV^2)"].values
err_data = data["dsigdt error (mu barn/GeV^2)"].values
t_data = -data["t"].values  # keep sign consistent

# Precompute anything reusable
s_data = W_data**2


# ============================
# MODEL WRAPPER (VECTORIZED)
# ============================
def sigma_model(W, params):
    """
    Vectorized wrapper.
    Assumes model.sig_omega can accept numpy arrays.
    """
    try:
        return model.sig_omega(W, params)
    except:
        # Fallback if model is NOT vectorized
        return np.array([model.sig_omega(w, params) for w in W])


# ============================
# OPTIONAL: PARALLEL VERSION
# ============================
def sigma_model_parallel(W, params):
    return np.array(
        Parallel(n_jobs=-1)(
            delayed(model.sig_omega)(w, params) for w in W
        )
    )


# ============================
# CHI-SQUARED FUNCTION
# ============================
def chi_squared(params):
    model_vals = sigma_model(W_data, params)

    # Protect against NaNs or invalid values
    if np.any(np.isnan(model_vals)) or np.any(np.isinf(model_vals)):
        return 1e20

    return np.sum(((sigma_data - model_vals) / err_data)**2)


# ============================
# INITIAL PARAMETERS
# ============================
initial_params = np.array([
    13.0, 6.0,        # g_pi_NN, g_eta_NN
    1.0, 1.0,         # Lambda_pi_NN, Lambda_eta_NN
    1.0, 1.0,         # Lambda_pi_gam_omega, Lambda_eta_gam_omega
    1.0, 1.0,         # Lambda_pi_gam_phi, Lambda_eta_gam_phi
    2.0, 2.0,         # b_P_omega_omega, b_P_phi_phi
    4.0, 4.0,         # B_omega, B_phi
    0.25,             # alpha_Pom_prime
    0.1, 0.1          # a_P_omega_omega, a_P_phi_phi
])


# ============================
# PARAMETER BOUNDS
# ============================
bounds = [(0, None)] * len(initial_params)


# ============================
# FITTING (FAST OPTIMIZER)
# ============================
print("Starting fit...")

result = minimize(
    chi_squared,
    initial_params,
    method='L-BFGS-B',
    bounds=bounds,
    options={
        'maxiter': 2000,
        'disp': True
    }
)

best_params = result.x


# ============================
# OUTPUT RESULTS
# ============================
print("\nBest-fit parameters:")
for i, val in enumerate(best_params):
    print(f"param[{i}] = {val}")

print(f"\nFinal chi^2 = {result.fun}")
print(f"Converged: {result.success}")


# ============================
# GENERATE FIT CURVE
# ============================
W_fit = np.linspace(min(W_data), max(W_data), 300)
sigma_fit = sigma_model(W_fit, best_params)


# ============================
# PLOT RESULTS
# ============================
plt.figure()

plt.errorbar(
    W_data,
    sigma_data,
    yerr=err_data,
    fmt='o',
    label="Data"
)

plt.plot(
    W_fit,
    sigma_fit,
    label="Model Fit"
)

plt.xlabel("W (GeV)")
plt.ylabel("Cross section σ (μb / GeV²)")
plt.title("Cross section σ vs W")
plt.legend()

plt.show()

Starting fit...
